# SORT1 locus — fine-map 11 LD proxies

Reproduces the MCP `fine_map_causal_variant` call for the lead
variant **rs12740374** at the SORT1 locus.

The 11 LD proxies (CEU, r²≥0.85) are inlined as a
Python list so the notebook runs offline. Pass them through
`prioritize_causal_variants` to get the composite ranking.

_To re-fetch from LDlink instead, uncomment the `fetch_ld_variants`
cell at the bottom._

In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.utils.ld import ld_variants_from_list
from chorus.analysis.causal import prioritize_causal_variants

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs — inline LD proxies (no LDlink network dep).
oracle_name = 'alphagenome'
lead_variant_id = 'rs12740374'
gene_name = 'SORT1'
r2_threshold = 0.85
ld_variants_raw = [
    {'id': 'rs12740374', 'chrom': 'chr1', 'pos': 109274968, 'ref': 'G', 'alt': 'T', 'r2': 1.0},
    {'id': 'rs4970836', 'chrom': 'chr1', 'pos': 109279175, 'ref': 'G', 'alt': 'A', 'r2': 0.907},
    {'id': 'rs1624712', 'chrom': 'chr1', 'pos': 109275908, 'ref': 'C', 'alt': 'T', 'r2': 1.0},
    {'id': 'rs660240', 'chrom': 'chr1', 'pos': 109275216, 'ref': 'T', 'alt': 'C', 'r2': 0.9509},
    {'id': 'rs142678968', 'chrom': 'chr1', 'pos': 109275536, 'ref': 'C', 'alt': 'T', 'r2': 0.9509},
    {'id': 'rs1626484', 'chrom': 'chr1', 'pos': 109275684, 'ref': 'G', 'alt': 'T', 'r2': 1.0},
    {'id': 'rs7528419', 'chrom': 'chr1', 'pos': 109274570, 'ref': 'A', 'alt': 'G', 'r2': 1.0},
    {'id': 'rs56960352', 'chrom': 'chr1', 'pos': 109278685, 'ref': 'G', 'alt': 'T', 'r2': 0.907},
    {'id': 'rs1277930', 'chrom': 'chr1', 'pos': 109279521, 'ref': 'G', 'alt': 'A', 'r2': 0.907},
    {'id': 'rs599839', 'chrom': 'chr1', 'pos': 109279544, 'ref': 'G', 'alt': 'A', 'r2': 0.907},
    {'id': 'rs602633', 'chrom': 'chr1', 'pos': 109278889, 'ref': 'T', 'alt': 'G', 'r2': 0.8582},
]
assay_ids = [
    'DNASE/EFO:0001187 DNase-seq/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPA genetically modified (insertion) using CRISPR targeting H. sapiens CEBPA/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPB/.',
    'CHIP_HISTONE/EFO:0001187 Histone ChIP-seq H3K27ac/.',
    'CAGE/hCAGE EFO:0001187/+',
    'CAGE/hCAGE EFO:0001187/-',
]

In [ ]:
# Convert the inline dicts to LDVariant objects, marking the lead as sentinel.
ld_variants = ld_variants_from_list(
    variants=ld_variants_raw,
    sentinel_id=lead_variant_id,
)
lead_dict = next(v for v in ld_variants_raw if v["id"] == lead_variant_id)
print(f"Sentinel: {lead_dict['id']} at {lead_dict['chrom']}:{lead_dict['pos']} {lead_dict['ref']}>{lead_dict['alt']}")
print(f"Proxies: {len(ld_variants) - 1}")

In [ ]:
# Score + rank each proxy by the composite causal score
# (max_effect + n_layers + convergence + ref_activity).
normalizer = get_normalizer(oracle_name=oracle_name)
result = prioritize_causal_variants(
    oracle=oracle,
    lead_variant=lead_dict,
    ld_variants=ld_variants,
    assay_ids=assay_ids,
    gene_name=gene_name,
    oracle_name=oracle_name,
    weights=None,
    normalizer=normalizer,
    analysis_request=None,
    snvs_only=False,
)

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - rs12740374_SORT1_locus_causal_report.html (interactive IGV report)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(result.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(result.to_dict(), indent=2, default=str)
)
try:
    result.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = result.to_html(output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_locus_causal_report.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


In [ ]:
# Optional: refresh LD proxies from LDlink (requires a token).
# import os
# from chorus.utils.ld import fetch_ld_variants
# ld_variants_live = fetch_ld_variants(
#     variant_id=lead_variant_id,
#     population="CEU",
#     r2_threshold=r2_threshold,
#     token=os.environ.get("LDLINK_TOKEN"),
#     timeout=30.0,
#     genome_build="grch38",
#     snvs_only=False,
# )

## What this notebook produced

- `example_output.md` — ranked table of LD proxies by composite causal score
- `example_output.json` — per-variant per-layer scores
- `example_output.tsv` — flat table view
- `rs12740374_SORT1_locus_causal_report.html` — interactive IGV report
